# LoRA Fine-Tuning a Small Language Model (Qwen2.5-0.5B-Instruct)
Parameter-efficient fine-tuning with **LoRA (PEFT)** on an instruction dataset, measuring
**eval loss and perplexity before vs. after** training. Runs on a free Colab T4 GPU in ~20–30 minutes.

**Runtime → Change runtime type → T4 GPU** before running.

In [ ]:
!pip install -q transformers peft datasets accelerate

In [ ]:
import math, torch
from datasets import load_dataset
from transformers import (AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments,
                          DataCollatorForLanguageModeling)
from peft import LoraConfig, get_peft_model

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
N_SAMPLES, MAX_LEN = 2000, 512
tok = AutoTokenizer.from_pretrained(MODEL_ID)
tok.pad_token = tok.pad_token or tok.eos_token
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto")
print(f"Params: {model.num_parameters()/1e6:.0f}M")

## 1. Data: instruction → response pairs, formatted with the model's chat template

In [ ]:
ds = load_dataset("yahma/alpaca-cleaned", split=f"train[:{N_SAMPLES}]").train_test_split(test_size=0.1, seed=42)

def fmt(ex):
    user = ex["instruction"] + (("\n\n" + ex["input"]) if ex["input"] else "")
    text = tok.apply_chat_template([{"role": "user", "content": user},
                                    {"role": "assistant", "content": ex["output"]}], tokenize=False)
    return tok(text, truncation=True, max_length=MAX_LEN)

train = ds["train"].map(fmt, remove_columns=ds["train"].column_names)
test = ds["test"].map(fmt, remove_columns=ds["test"].column_names)
collator = DataCollatorForLanguageModeling(tok, mlm=False)
print(len(train), "train /", len(test), "eval examples")

## 2. Baseline: eval loss and perplexity *before* fine-tuning

In [ ]:
args = TrainingArguments("out", per_device_train_batch_size=8, per_device_eval_batch_size=8,
                         gradient_accumulation_steps=2, num_train_epochs=1, learning_rate=2e-4,
                         warmup_ratio=0.05, logging_steps=20, fp16=True, report_to="none",
                         save_strategy="no")
base = Trainer(model=model, args=args, eval_dataset=test, data_collator=collator).evaluate()
print(f"BEFORE  eval_loss={base['eval_loss']:.3f}  perplexity={math.exp(base['eval_loss']):.2f}")

## 3. LoRA: train only ~1% of parameters (rank-16 adapters on attention projections)

In [ ]:
model = get_peft_model(model, LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
                                         target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]))
model.print_trainable_parameters()
trainer = Trainer(model=model, args=args, train_dataset=train, eval_dataset=test, data_collator=collator)
trainer.train()

## 4. Results: eval loss and perplexity *after* fine-tuning

In [ ]:
after = trainer.evaluate()
b, a = base["eval_loss"], after["eval_loss"]
print(f"BEFORE  eval_loss={b:.3f}  perplexity={math.exp(b):.2f}")
print(f"AFTER   eval_loss={a:.3f}  perplexity={math.exp(a):.2f}")
print(f"Perplexity reduced by {100*(1-math.exp(a)/math.exp(b)):.1f}%")

## 5. Qualitative check

In [ ]:
model.eval()
for q in ["Give three tips for staying focused while studying.", "Explain what an embedding is in one paragraph."]:
    ids = tok.apply_chat_template([{"role": "user", "content": q}], add_generation_prompt=True, return_tensors="pt").to(model.device)
    out = model.generate(ids, max_new_tokens=150, do_sample=False)
    print("Q:", q, "\nA:", tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True), "\n")

In [ ]:
model.save_pretrained("qwen2.5-0.5b-alpaca-lora")   # small adapter (~10–20 MB)
tok.save_pretrained("qwen2.5-0.5b-alpaca-lora")